In [2]:
import pandas as pd
import numpy as np
pd.options.display.max_rows = 999

In [37]:
class BuySell:

    def __init__(self, capital):
        self.initial_capital = capital
        self.current_capital = capital
        self.share_amount = 0


    def buyandhold(self, dataframe):
        """

        Args:
            dataframe: (pd.DataFrame) should contains price and directive, date columns

        Returns: (float) profit

        """
        first_price = dataframe.iloc[0].loc['price']
        last_price  = dataframe.iloc[-1].loc['price']

        self.current_capital, self.share_amount = BuySell._buy(self.initial_capital, first_price)
        money, self.share_amount = BuySell._sell(self.share_amount, last_price)
        self.current_capital += money

        profit = self.current_capital - self.initial_capital

        return profit

    @staticmethod
    def _buy(money, price):
        """buys if appliable
        Returns: (float, int) current_money, share_amount"""

        remaining_money = money
        share_amount = 0

        if money > price:
            share_amount = int(money / price)
            remaining_money = money - share_amount * price

        return remaining_money, share_amount

    @staticmethod
    def _sell(share_amount, price):
        """sells all shares :)
        Returns: (float, int) current_money, share_amount"""
        money = share_amount * price
        return money, 0



    def process(self, dataframe):
        """

        Args:
            dataframe: (pd.DataFrame) should contains price and directive, date columns

        Returns: (float) profit

        """
        capital_list = list()
        share_amount_list = list()

        for idx,row in dataframe.iterrows():
            if row['directive'] == 'buy':
                remaining_money, share_amount = BuySell._buy(self.current_capital, row['price'])
                self.current_capital = remaining_money
                self.share_amount += share_amount

            if row['directive'] == 'sell':
                money, share_amount = BuySell._sell(self.share_amount, row['price'])
                self.current_capital += money
                self.share_amount = share_amount
            capital_list.append(self.current_capital)
            share_amount_list.append(self.share_amount)

        dataframe.loc[:,'current_capital'] = capital_list
        dataframe.loc[:,'share_amount'] = share_amount_list
        dataframe.loc[:,'total_capital'] = dataframe['current_capital'] + dataframe['share_amount']*dataframe['price']

        dataframe.loc[:,'profit'] = dataframe['total_capital'] - self.initial_capital


        return dataframe


In [4]:
def label_to_directives(row):
    row = row[['pbuy','psell','phold']]
    argmax_idx = np.argmax(row.values)

    if argmax_idx == 0:
        return 'buy'
    if argmax_idx == 1:
        return 'sell'
    return 'hold'


In [10]:
# np.random.seed(42)
# size = 100
# fake_dataframe = pd.DataFrame({'price':np.random.randint(100,150, size=size),
#                                'directive':np.random.choice(['buy','hold','sell'], size=size, replace=True),
#      'name':np.random.choice(['xlp','xlu','xlv','xly','dia','ewa','ewc','ewg','ewh','ewj','eww','spy','xlb','xle','xlf','xli','xlk'], size=size, replace=True)})


In [41]:
initial_capital = 100000
stock_names = ['xlp','xlu','xlv','xly','dia','ewa','ewc','ewg','ewh','ewj','eww','spy','xlb','xle','xlf','xli','xlk']
exp_name = 'exp2'

final_result_df = None
for stock_name in stock_names:
    dataframe = pd.read_csv('../experiment/finance_cnn_{}_{}_only/prediction_results.csv'.format(exp_name,stock_name))
    dataframe['directive'] = dataframe.apply(label_to_directives, axis=1)
    dataframe['price'] = dataframe['raw_adjusted_close'].values
    
    # Buy and Hold strategy
    buyandhold_profit = BuySell(capital=initial_capital).buyandhold(dataframe)
    
    # Our simple strategy
    buysell = BuySell(capital=initial_capital)
    buysell_result_df = buysell.process(dataframe)
#     subset = ['name','date','price','directive','current_capital','share_amount','total_capital', 'profit']
    subset = buysell_result_df.columns
    if final_result_df is None:
        final_result_df = pd.DataFrame(columns=subset)
    
    last_row = buysell_result_df[subset].iloc[-1]
    last_row['bah_profit'] = buyandhold_profit
    
    final_result_df = final_result_df.append(last_row)
    
final_result_df = final_result_df.reset_index(drop=True)



48.13 50.45
42.97 47.43
74.28 70.27
76.56 79.72
175.64 180.84
21.85 20.28
27.53 25.47
29.56 25.24
23.02 21.4
51.6 49.39
58.36 44.12
204.53 216.59
49.62 48.15
77.09 69.32
24.94 19.49
55.92 56.95
41.44 47.42


In [42]:
final_result_df

,psell,pbuy,phold,rsell,rbuy,rhold,date,name,raw_adjusted_close,directive,price,current_capital,share_amount,total_capital,profit,bah_profit
0,-0.127310,0.362543,0.763712,0.0,0.0,1.0,2016-11-14,xlp,50.45,hold,50.45,49.56,2076,104783.76,4783.76,4818.64
1,0.123012,0.073775,0.809242,0.0,0.0,1.0,2016-10-12,xlu,47.43,hold,47.43,135125.76,0,135125.76,35125.76,10378.42
2,0.049328,0.562109,0.363915,0.0,0.0,1.0,2016-11-14,xlv,70.27,buy,70.27,53.25,1542,108409.59,8409.59,-5397.46
3,0.309237,0.254832,0.379430,0.0,1.0,0.0,2016-11-11,xly,79.72,hold,79.72,50.38,1340,106875.18,6875.18,4126.96
4,0.212411,0.374944,0.374252,0.0,0.0,1.0,2016-10-13,dia,180.84,buy,180.84,11.95,625,113036.95,13036.95,2958.80
5,0.078466,0.304138,0.639551,0.0,0.0,1.0,2016-10-31,ewa,20.28,hold,20.28,135223.44,0,135223.44,35223.44,-7184.32
6,-0.130721,0.573570,0.632052,0.0,0.0,1.0,2016-10-26,ewc,25.47,hold,25.47,15.28,4704,119826.16,19826.16,-7481.92
7,0.023147,0.090994,0.865849,0.0,0.0,1.0,2016-11-14,ewg,25.24,hold,25.24,5.73,4341,109572.57,9572.57,-14610.24
8,0.347520,0.351721,0.340014,1.0,0.0,0.0,2016-11-03,ewh,21.40,buy,21.40,1.27,4754,101736.87,1736.87,-7037.28
9,0.328698,-0.132670,0.860548,0.0,1.0,0.0,2016-11-09,ewj,49.39,hold,49.39,105893.28,0,105893.28,5893.28,-4280.77
